https://doc.rust-lang.org/reference/linkage.html
https://doc.rust-lang.org/nomicon/ffi.html#calling-rust-code-from-c
--crate-type

https://google.github.io/comprehensive-rust/unsafe-rust/unsafe.html
https://doc.rust-lang.org/book/ch20-01-unsafe-rust.html
https://doc.rust-lang.org/nomicon/

In [35]:
%%file /tmp/ptr.rs

extern "C" {static mut foo : i32;}

#[unsafe(no_mangle)]
extern "C" fn flum(){
    unsafe{
        foo += 1;
    }
}


Overwriting /tmp/ptr.rs


In [36]:
! rustc -g --crate-type=staticlib -o /tmp/libptr.a /tmp/ptr.rs

In [37]:
%%file /tmp/test.c
#include <stdint.h>
#include <stdio.h>
uint32_t foo = 0;
void flum();

int main() {
    flum();
    printf("%d\n", foo);
}

Overwriting /tmp/test.c


In [40]:
! gcc -Wextra -Wall -g  -o /tmp/test /tmp/test.c  -L/tmp -lptr && /tmp/test

1


In [71]:
%%file /tmp/ptr.rs
fn main() {
    let mut x : i32 = 10;

    let p1: *mut i32 = &raw mut x;
    let p2 = p1 as *const i32;
    unsafe {
        dbg!(*p1);
        dbg!(std::mem::size_of_val(&x));
        dbg!(std::mem::size_of_val(&p1));
        dbg!(std::mem::size_of::<*mut i32>());
        dbg!(std::mem::size_of::<&i32>());
        dbg!(std::mem::size_of::<&str>());
        *p1 = 6;
        // Mutation may soundly be observed through a raw pointer, like in C.
        dbg!(*p2);
    }
}

Overwriting /tmp/ptr.rs


In [72]:
!rustc -g -o /tmp/ptr /tmp/ptr.rs && /tmp/ptr

[/tmp/ptr.rs:7:9] *p1 = 10
[/tmp/ptr.rs:8:9] std::mem::size_of_val(&x) = 4
[/tmp/ptr.rs:9:9] std::mem::size_of_val(&p1) = 8
[/tmp/ptr.rs:10:9] std::mem::size_of::<*mut i32>() = 8
[/tmp/ptr.rs:11:9] std::mem::size_of::<&i32>() = 8
[/tmp/ptr.rs:12:9] std::mem::size_of::<&str>() = 16
[/tmp/ptr.rs:15:9] *p2 = 6


In [ ]:
%%file /tmp/array.rs

static mut FOO : [u32; 128] = [0 ; 128];

fn flum(ptr : *mut u32, len : usize){
    unsafe {
        for i in 0..len {
            (*ptr).add(i) += 1;
        }
    }
}

fn main() {
    let ptr = (&raw mut FOO).cast::<u32>();
    flum(ptr, 129);
    unsafe{ dbg!(FOO) };
}


Overwriting /tmp/array.rs


In [101]:
! rustc -g -o /tmp/array /tmp/array.rs && /tmp/array

[/tmp/array.rs:15:13] FOO = [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
]


In [97]:
! rustc +nightly -Zsanitizer=address -g -o /tmp/array-asan /tmp/array.rs && /tmp/array-asan

==2611300==ERROR: AddressSanitizer: global-buffer-overflow on address 0x603d67352a60 at pc 0x603d66dda980 bp 0x7fff8c0e78d0 sp 0x7fff8c0e78c8
READ of size 4 at 0x603d67352a60 thread T0
Traceback (most recent call last):
  File "/home/philip/philzook58.github.io/.venv/bin/llvm-symbolizer", line 4, in <module>
    from _mojo._entrypoints import exec_llvm_symbolizer
ModuleNotFoundError: No module named '_mojo'
==2611300==WARNING: Can't read from symbolizer at fd 3
==2611300==WARNING: Can't write to symbolizer at fd 6
Traceback (most recent call last):
  File "/home/philip/philzook58.github.io/.venv/bin/llvm-symbolizer", line 4, in <module>
    from _mojo._entrypoints import exec_llvm_symbolizer
ModuleNotFoundError: No module named '_mojo'
==2611300==WARNING: Can't read from symbolizer at fd 3
==2611300==WARNING: Can't write to symbolizer at fd 6
Traceback (most recent call last):
  File "/home/philip/philzook58.github.io/.venv/bin/llvm-symbolizer", line 4, in <module>
    from _mojo._entr